In [79]:
#Importacion de librerias
import requests
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures

#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [ ]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 

        
        SELECT DISTINCT * FROM GRADOS.Consolidado_Grados
        WHERE NIVEL = 'PROFESIONAL'

 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_grados = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

### ESPECIALIZACIONES PARA VER QUE ESTUDIANTES REALIZARON RECOMPRA

In [36]:
df_recompra_profesional = df_grados.copy()

### MALLAS PARA LLEVAR EL COD_UNIDAD

### ESTADISTICA PARA VER DIPLOMADOS Y ESPECIALIZACIONES MATRICULADAS

In [ ]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 

        
         SELECT DISTINCT NUM_IDENTIFICACION as NUMERO,NIVEL_FORMACION,NOM_DEPENDENCIA,COD_UNIDAD, COD_PERIODO,DEPARTAMENTO_SEDE AS DEPARTAMENTO_ESPECIALIZACION ,MUNICIPIO_SEDE AS SEDE_ESPECIALIZACION, AÑO AS AÑO_EST
        FROM CUN.ESTADISTICA_ESTUDIANTE_2
        where (NIVEL_FORMACION = 'Especializacion'
        or NIVEL_FORMACION = 'Diplomado')

 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_est = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

#### SE CREA DF PARA SABER SI MATRICULARON Y NO ESTAN GRADUADOS

In [72]:
df_esp_matriculado = df_est.copy()

In [73]:
df_esp_matricula_1 = df_esp_matriculado[df_esp_matriculado['NUMERO'].isin(df_recompra_profesional['NUMERO'])]
df_esp_matricula_1

,NUMERO,NIVEL_FORMACION,NOM_DEPENDENCIA,COD_UNIDAD,COD_PERIODO,DEPARTAMENTO_ESPECIALIZACION,SEDE_ESPECIALIZACION
3,1000000929,Diplomado,DIPLOMADO GERENCIA DEL TALENTO HUMANO,CE152,2020E,BOGOTA D.C,Bogota
4,1000001280,Diplomado,DIPLOMADO INTELIGENCIA ARTIFICIAL PARA LA RESO...,CE418,2023E,BOGOTA D.C,Bogota
9,1000001855,Diplomado,DIPLOMADO BIG DATA,CE421,2023E,BOGOTA D.C,Bogota
10,1000002891,Diplomado,DIPLOMADO DESARROLLO DE APLICACIONES DE SOTFWA...,CE266,2023E,BOGOTA D.C,Bogota
11,1000003486,Diplomado,DIPLOMADO GERENCIA DEL TALENTO HUMANO,CE277,24E03,BOGOTA D.C,Bogota
...,...,...,...,...,...,...,...
59551,98765710,Especializacion,ESPECIALIZACIÓN EN TRANSFORMACIÓN DIGITAL,ESP30,26ES1,BOGOTA D.C,Bogota
59557,9911460,Especializacion,ESPECIALIZACIÓN EN ALTA GERENCIA,ESP24,26ES3,BOGOTA D.C,Bogota
59563,9956823,Diplomado,DIPLOMADO GERENCIA DE PROYECTOS ALINEADO CON L...,CE151,2022E,BOGOTA D.C,Bogota
59566,9975883,Diplomado,ALTA DIRECCIÓN EN GESTIÓN Y LIDERAZGO ESTRATE...,CER02,2016E,RISARALDA,Pereira


In [74]:
df_final_especializaciones = df_recompra_profesional.merge(df_esp_matricula_1, how='inner',on='NUMERO')
df_final_especializaciones.drop_duplicates(inplace=True)
df_final_especializaciones.shape

(23263, 44)

#### CRUZAR PARA TENER EN CUENTA LAS ESPECIALIZACIONES MATRICULADAS

In [76]:
df_fin = df_final_especializaciones[['Nombres', 'genero_code', 'tipo_doc_unico', 'NUMERO', 'grad_annio',
       'grad_semestre', 'PLAN', 'snies', 'METODOLOGIA', 'NIVEL', 'TITULACION',
       'ESCUELA', 'fecha_grado', 'numero_de_credencial',
       'codigo_municipio_grado', 'SEDE_GRADO', 'acta', 'No_de_Registro',
       'folio', 'libro', 'ciclo_grado', 'Celular_Sugerido',
       'Telefono_Sugerido', 'Otros_Sugeridos', 'Correo_Personal_Sugerido',
       'PROGRAMA_PROFESIONAL', 'NOM_PROGRAMA_PROFESIONAL', 'SEDE_GRADO_2', 'COD_PERIODO','DEPARTAMENTO_ESPECIALIZACION' ,'SEDE_ESPECIALIZACION',
       'SCORE_M0', 'opcion_grado','NOM_DEPENDENCIA','NIVEL_FORMACION']]

df_fin

,Nombres,genero_code,tipo_doc_unico,NUMERO,grad_annio,grad_semestre,PLAN,snies,METODOLOGIA,NIVEL,...,PROGRAMA_PROFESIONAL,NOM_PROGRAMA_PROFESIONAL,SEDE_GRADO_2,COD_PERIODO,DEPARTAMENTO_ESPECIALIZACION,SEDE_ESPECIALIZACION,SCORE_M0,opcion_grado,NOM_DEPENDENCIA,NIVEL_FORMACION
0,FABIAN ANDRES LOPEZ GIL,o,CC,1023963637,2024,1,VG001,104396,VIRTUAL,PROFESIONAL,...,INGENIERÍA DE SISTEMAS,INGENIERÍA DE SISTEMAS,"Bogotá, D.C.",2022E,BOGOTA D.C,Bogota,No Contactado,HOMOLOGACION DIPLOMADO,DIPLOMADO MACHINE LEARNING EN PYTHON,Diplomado
1,AARON SANTIAGO SUAREZ ABELLO,o,CC,1007365381,2025,1,1037,108556,VIRTUAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",24ET3,BOGOTA D.C,Bogota,None,None,ESPECIALIZACIÓN EN GERENCIA DE LA MARCA,Especializacion
2,AARON SANTIAGO SUAREZ ABELLO,o,CC,1007365381,2025,1,1037,108556,VIRTUAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",24ET4,BOGOTA D.C,Bogota,None,None,ESPECIALIZACIÓN EN GERENCIA DE LA MARCA,Especializacion
3,AARON SANTIAGO SUAREZ ABELLO,o,CC,1007365381,2025,1,1037,108556,VIRTUAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",25ET1,BOGOTA D.C,Bogota,None,None,ESPECIALIZACIÓN EN GERENCIA DE LA MARCA,Especializacion
4,AARON SANTIAGO SUAREZ ABELLO,o,CC,1007365381,2025,1,1037,108556,VIRTUAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",25ET2,BOGOTA D.C,Bogota,None,None,ESPECIALIZACIÓN EN GERENCIA DE LA MARCA,Especializacion
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23258,ZUÑIGA AMU YAISA XIMENA,a,CC,1111810542,2021,2,1028,52612,DISTANCIA,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,ADMINISTRACIÓN DE EMPRESAS,Buenaventura,2019E,BOGOTA D.C,Bogota,No Contactado,Homologación,DIPLOMADO DESARROLLO SOSTENIBLE - UNA MIRADA A...,Diplomado
23259,ZUÑIGA ARRIETA BRAYAN DAVID,o,CC,1001270769,2024,2,VG001,104396,VIRTUAL,PROFESIONAL,...,INGENIERÍA DE SISTEMAS,INGENIERÍA DE SISTEMAS,"Bogotá, D.C.",2023E,BOGOTA D.C,Bogota,Contactado,None,DIPLOMADO DESARROLLO DE APLICACIONES DE SOTFWA...,Diplomado
23260,ZUÑIGA ARRIETA YONATHAN ALEXANDER,o,CC,1001276672,2024,2,VG001,104396,VIRTUAL,PROFESIONAL,...,INGENIERÍA DE SISTEMAS,INGENIERÍA DE SISTEMAS,"Bogotá, D.C.",2023E,BOGOTA D.C,Bogota,Contactado,None,DIPLOMADO DESARROLLO DE APLICACIONES DE SOTFWA...,Diplomado
23261,ZUÑIGA PLAZA VICTORIA EUGENIA,a,CC,45487988,2024,2,1037,108556,VIRTUAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",2023E,BOGOTA D.C,Bogota,Contactado,None,DIPLOMADO DESARROLLO SOSTENIBLE - UNA MIRADA A...,Diplomado


In [81]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 

        
         SELECT distinct PERIODO as COD_PERIODO,
                COALESCE(ANIO,
                         TRY_CAST(LEFT(PERIODO, 4) AS BIGINT),
                         TRY_CAST('2' + LEFT(PERIODO, 3) AS BIGINT),
                         TRY_CAST('20' + LEFT(PERIODO, 2) AS BIGINT)
                ) ANIO
FROM ZOHO.Base_Personas


 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_año = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [83]:
df_final_final = df_fin.merge(df_año,how='left',on='COD_PERIODO')
df_final_final

,Nombres,genero_code,tipo_doc_unico,NUMERO,grad_annio,grad_semestre,PLAN,snies,METODOLOGIA,NIVEL,...,NOM_PROGRAMA_PROFESIONAL,SEDE_GRADO_2,COD_PERIODO,DEPARTAMENTO_ESPECIALIZACION,SEDE_ESPECIALIZACION,SCORE_M0,opcion_grado,NOM_DEPENDENCIA,NIVEL_FORMACION,ANIO
0,FABIAN ANDRES LOPEZ GIL,o,CC,1023963637,2024,1,VG001,104396,VIRTUAL,PROFESIONAL,...,INGENIERÍA DE SISTEMAS,"Bogotá, D.C.",2022E,BOGOTA D.C,Bogota,No Contactado,HOMOLOGACION DIPLOMADO,DIPLOMADO MACHINE LEARNING EN PYTHON,Diplomado,2022
1,AARON SANTIAGO SUAREZ ABELLO,o,CC,1007365381,2025,1,1037,108556,VIRTUAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",24ET3,BOGOTA D.C,Bogota,None,None,ESPECIALIZACIÓN EN GERENCIA DE LA MARCA,Especializacion,2024
2,AARON SANTIAGO SUAREZ ABELLO,o,CC,1007365381,2025,1,1037,108556,VIRTUAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",24ET4,BOGOTA D.C,Bogota,None,None,ESPECIALIZACIÓN EN GERENCIA DE LA MARCA,Especializacion,2024
3,AARON SANTIAGO SUAREZ ABELLO,o,CC,1007365381,2025,1,1037,108556,VIRTUAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",25ET1,BOGOTA D.C,Bogota,None,None,ESPECIALIZACIÓN EN GERENCIA DE LA MARCA,Especializacion,2025
4,AARON SANTIAGO SUAREZ ABELLO,o,CC,1007365381,2025,1,1037,108556,VIRTUAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",25ET2,BOGOTA D.C,Bogota,None,None,ESPECIALIZACIÓN EN GERENCIA DE LA MARCA,Especializacion,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23258,ZUÑIGA AMU YAISA XIMENA,a,CC,1111810542,2021,2,1028,52612,DISTANCIA,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,Buenaventura,2019E,BOGOTA D.C,Bogota,No Contactado,Homologación,DIPLOMADO DESARROLLO SOSTENIBLE - UNA MIRADA A...,Diplomado,2019
23259,ZUÑIGA ARRIETA BRAYAN DAVID,o,CC,1001270769,2024,2,VG001,104396,VIRTUAL,PROFESIONAL,...,INGENIERÍA DE SISTEMAS,"Bogotá, D.C.",2023E,BOGOTA D.C,Bogota,Contactado,None,DIPLOMADO DESARROLLO DE APLICACIONES DE SOTFWA...,Diplomado,2023
23260,ZUÑIGA ARRIETA YONATHAN ALEXANDER,o,CC,1001276672,2024,2,VG001,104396,VIRTUAL,PROFESIONAL,...,INGENIERÍA DE SISTEMAS,"Bogotá, D.C.",2023E,BOGOTA D.C,Bogota,Contactado,None,DIPLOMADO DESARROLLO DE APLICACIONES DE SOTFWA...,Diplomado,2023
23261,ZUÑIGA PLAZA VICTORIA EUGENIA,a,CC,45487988,2024,2,1037,108556,VIRTUAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",2023E,BOGOTA D.C,Bogota,Contactado,None,DIPLOMADO DESARROLLO SOSTENIBLE - UNA MIRADA A...,Diplomado,2023


In [91]:
df_final_2 = df_final_final.sort_values(['ANIO'], ascending=True)\
                   .drop_duplicates(subset=['NUMERO','NIVEL_FORMACION'], keep='first')
df_final_2

,Nombres,genero_code,tipo_doc_unico,NUMERO,grad_annio,grad_semestre,PLAN,snies,METODOLOGIA,NIVEL,...,NOM_PROGRAMA_PROFESIONAL,SEDE_GRADO_2,COD_PERIODO,DEPARTAMENTO_ESPECIALIZACION,SEDE_ESPECIALIZACION,SCORE_M0,opcion_grado,NOM_DEPENDENCIA,NIVEL_FORMACION,ANIO
6480,EDWIN EDICSON CAÑON TORRES,o,CC,11510597,2014,1,1014,52612,DISTANCIA,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",2013B,BOGOTA D.C,Bogota,No Contactado,None,DIPLOMADO GERENCIA DE MERCADEO,Diplomado,2013
10627,JOHANNA ANDREA MORENO,a,CC,53132767,2014,1,1011,52605,PRESENCIAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",2013B,BOGOTA D.C,Bogota,No Contactado,CURSO NORMAL,DIPLOMADO GERENCIA DE MERCADEO,Diplomado,2013
2001,ARMANDO JUVENAL CASTIBLANCO PALACIOS,o,CC,11521295,2014,1,1014,52612,DISTANCIA,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",2013B,BOGOTA D.C,Bogota,No Contactado,CURSO NORMAL,DIPLOMADO GERENCIA DE MERCADEO,Diplomado,2013
15371,MANUEL HERNANDO RODRIGUEZ RODRIGUEZ,o,CC,80164143,2014,1,1011,52605,PRESENCIAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",2013B,BOGOTA D.C,Bogota,No Contactado,Homologación,DIPLOMADO GERENCIA DE MERCADEO,Diplomado,2013
6800,ENID JOANNA HERNANDEZ RIOS,a,CC,38360307,2013,1,1011,52605,PRESENCIAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",2013B,BOGOTA D.C,Bogota,No Contactado,CURSO NORMAL,DIPLOMADO GERENCIA DE MERCADEO,Diplomado,2013
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10662,JOHANNA PATRICIA ESPINOSA AGUDELO,a,CC,52714131,2025,2,1037,108556,VIRTUAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",26ES1,BOGOTA D.C,Bogota,None,None,ESPECIALIZACIÓN EN ALTA GERENCIA,Especializacion,2026
14546,LUIS ALBERTO PEREZ BARBOSA,o,CC,1082976540,2021,1,8013,53501,PRESENCIAL,PROFESIONAL,...,NEGOCIOS INTERNACIONALES,Santa Marta,26ES1,BOGOTA D.C,Bogota,No Contactado,CURSO NORMAL,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,Especializacion,2026
7368,FERNANDO ALEXIS HERNANDEZ ROMERO,o,CC,1024487528,2025,2,1073,108486,VIRTUAL,PROFESIONAL,...,CONTADURÍA PÚBLICA,"Bogotá, D.C.",26ES1,BOGOTA D.C,Bogota,None,None,ESPECIALIZACIÓN EN GERENCIA DE LA MARCA,Especializacion,2026
7373,FERNANDO ANTONIO PEÑA PEÑA,M,CC,1082835487,2025,2,1037,108556,VIRTUAL,PROFESIONAL,...,ADMINISTRACIÓN DE EMPRESAS,"Bogotá, D.C.",26ES1,BOGOTA D.C,Bogota,None,None,ESPECIALIZACIÓN EN ALTA GERENCIA,Especializacion,2026


In [94]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(''' 

        
         SELECT Identificacion as NUMERO, lat, lon, ciudad, departamento  FROM CUN_REPOSITORIO.GEO.estadistica_geolocalizacion

 ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_ciudad = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [96]:
df_final_3 = df_final_2.merge(df_ciudad, how='left', on='NUMERO')
df_final_3

,Nombres,genero_code,tipo_doc_unico,NUMERO,grad_annio,grad_semestre,PLAN,snies,METODOLOGIA,NIVEL,...,SEDE_ESPECIALIZACION,SCORE_M0,opcion_grado,NOM_DEPENDENCIA,NIVEL_FORMACION,ANIO,lat,lon,ciudad,departamento
0,EDWIN EDICSON CAÑON TORRES,o,CC,11510597,2014,1,1014,52612,DISTANCIA,PROFESIONAL,...,Bogota,No Contactado,None,DIPLOMADO GERENCIA DE MERCADEO,Diplomado,2013,NaN,NaN,NaN,NaN
1,JOHANNA ANDREA MORENO,a,CC,53132767,2014,1,1011,52605,PRESENCIAL,PROFESIONAL,...,Bogota,No Contactado,CURSO NORMAL,DIPLOMADO GERENCIA DE MERCADEO,Diplomado,2013,NaN,NaN,NaN,NaN
2,ARMANDO JUVENAL CASTIBLANCO PALACIOS,o,CC,11521295,2014,1,1014,52612,DISTANCIA,PROFESIONAL,...,Bogota,No Contactado,CURSO NORMAL,DIPLOMADO GERENCIA DE MERCADEO,Diplomado,2013,4.756153,-74.050482,BOGOTA,DISTRITO CAPITAL
3,MANUEL HERNANDO RODRIGUEZ RODRIGUEZ,o,CC,80164143,2014,1,1011,52605,PRESENCIAL,PROFESIONAL,...,Bogota,No Contactado,Homologación,DIPLOMADO GERENCIA DE MERCADEO,Diplomado,2013,NaN,NaN,NaN,NaN
4,ENID JOANNA HERNANDEZ RIOS,a,CC,38360307,2013,1,1011,52605,PRESENCIAL,PROFESIONAL,...,Bogota,No Contactado,CURSO NORMAL,DIPLOMADO GERENCIA DE MERCADEO,Diplomado,2013,2.432262,-76.604804,POPAYAN,CAUCA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16813,JOHANNA PATRICIA ESPINOSA AGUDELO,a,CC,52714131,2025,2,1037,108556,VIRTUAL,PROFESIONAL,...,Bogota,None,None,ESPECIALIZACIÓN EN ALTA GERENCIA,Especializacion,2026,4.585568,-74.077077,BOGOTA,DISTRITO CAPITAL
16814,LUIS ALBERTO PEREZ BARBOSA,o,CC,1082976540,2021,1,8013,53501,PRESENCIAL,PROFESIONAL,...,Bogota,No Contactado,CURSO NORMAL,ESPECIALIZACIÓN EN GERENCIA DE PROYECTOS,Especializacion,2026,11.201689,-74.224916,SANTA MARTA,MAGDALENA
16815,FERNANDO ALEXIS HERNANDEZ ROMERO,o,CC,1024487528,2025,2,1073,108486,VIRTUAL,PROFESIONAL,...,Bogota,None,None,ESPECIALIZACIÓN EN GERENCIA DE LA MARCA,Especializacion,2026,4.728482,-74.230779,MOSQUERA,CUNDINAMARCA
16816,FERNANDO ANTONIO PEÑA PEÑA,M,CC,1082835487,2025,2,1037,108556,VIRTUAL,PROFESIONAL,...,Bogota,None,None,ESPECIALIZACIÓN EN ALTA GERENCIA,Especializacion,2026,9.787420,-74.776620,PLATO,MAGDALENA


In [99]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'RECOMPRA3'
schema_name = 'SRUBIO'

# Send the data to sql server
df_final_3.to_sql(
    name=table_name,
    schema=schema_name, 
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()